# 🏦 Magentic Multi-Agent Orchestration for Investment Research

## Overview

This notebook demonstrates **Magentic orchestration** for financial services - an advanced multi-agent coordination pattern for comprehensive investment research and analysis. The orchestrator automatically manages research task decomposition, specialist agent selection, and insight synthesis.

> ⚠️ **Financial Services Disclaimer**: This notebook demonstrates AI agent workflows for educational purposes. In production, all investment research and recommendations must comply with regulatory requirements (SEC, FINRA) and include appropriate disclosures.

### Industry Use Case: Investment Research Report

A wealth management firm needs to prepare quarterly investment research reports that require:
- **Market Research**: Current market conditions, sector trends
- **Quantitative Analysis**: Financial calculations, risk metrics
- **Report Synthesis**: Consolidated insights and recommendations

### Key Concepts

| Concept | Description |
|---------|-------------|
| **MagenticBuilder** | High-level API for multi-agent orchestration |
| **Standard Manager** | Built-in orchestrator with planning capabilities |
| **Specialized Agents** | Domain experts (Market Researcher, Quant Analyst) |
| **Streaming Callbacks** | Real-time event monitoring |
| **Event Types** | Orchestrator messages, agent deltas, agent messages, final result |

### Architecture

```
Investment Research Request
           ↓
Magentic Orchestrator (Standard Manager)
           ↓
Research Plan Generation
           ↓
┌─────────────────────────────────┐
│     Delegate to Specialists     │
│  ├── MarketResearcherAgent      │
│  │   (Market data, sector news) │
│  └── QuantAnalystAgent          │
│      (Financial calculations)   │
└─────────────────────────────────┘
           ↓
Monitor & Adapt Plan
           ↓
Synthesize Investment Report
```

### Workflow Parameters

| Parameter | Default | Purpose |
|-----------|---------|---------|
| `max_round_count` | 10 | Maximum orchestration rounds |
| `max_stall_count` | 3 | Retries when progress stalls |
| `max_reset_count` | 2 | Full plan resets allowed |

## Prerequisites

- ✅ Azure AI Foundry configured
- ✅ Environment variables: `AI_FOUNDRY_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- ✅ Azure CLI authentication: Run `az login`

## 1️⃣ Setup and Imports

In [1]:
import asyncio
import os
from dotenv import load_dotenv
from typing import cast

from azure.identity import AzureCliCredential

from agent_framework import (
    Agent,
    AgentResponseUpdate,
    Message,
    WorkflowBuilder,
    WorkflowEvent,
)
from agent_framework.foundry import FoundryChatClient

# Load environment variables from .env file
load_dotenv('../../.env')

# Verify environment is loaded
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
print(f"✅ Environment loaded: {project_endpoint is not None and model_deployment is not None}")

✅ Environment loaded: True


## 2️⃣ Create Foundry Chat Client

We'll use a single Foundry client for all agents to share credentials and endpoint configuration.

In [2]:
# Create shared Foundry Chat Client
chat_client = FoundryChatClient(
    credential=AzureCliCredential(),
    project_endpoint=os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT"),
    model=os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
)

print("✅ Foundry Chat Client created")

✅ Foundry Chat Client created


## 3️⃣ Create Specialized Financial Agents

### MarketResearcherAgent
- Specializes in market data, sector trends, and financial news
- Gathers factual information about market conditions

### QuantAnalystAgent
- Performs financial calculations and risk analysis
- Provides detailed analysis with risk metrics and projections

In [3]:
# Create Market Researcher Agent
market_researcher = Agent(
    client=chat_client,
    name="MarketResearcherAgent",
    description="Specialist in market research, sector analysis, and financial news gathering",
    instructions=(
        "You are a Financial Market Researcher at a wealth management firm. "
        "You gather market data, sector trends, company news, and economic indicators. "
        "Focus on factual information without performing calculations. "
        "Always cite sources and note the date of any market data you reference."
    ),
)

# Create Quantitative Analyst Agent
quant_analyst = Agent(
    client=chat_client,
    name="QuantAnalystAgent",
    description="A quantitative analyst that performs financial calculations, risk analysis, and portfolio modeling.",
    instructions=(
        "You are a Quantitative Analyst at a wealth management firm. "
        "You perform financial calculations including: portfolio analysis, risk metrics (beta, Sharpe ratio, VaR), "
        "return calculations, and investment projections. "
        "Always show your calculation methodology and assumptions. "
        "Present results in clear tables when appropriate."
    ),
)

# Create Manager Agent to orchestrate the workflow
manager_agent = Agent(
    client=chat_client,
    name="ResearchManager",
    description="Orchestrator that coordinates the investment research workflow",
    instructions=(
        "You coordinate a team of financial analysts to complete investment research tasks efficiently. "
        "Review outputs from the Market Researcher and pass relevant findings to the Quant Analyst. "
        "Synthesize a final comprehensive investment research report. "
        "Ensure all research includes appropriate risk disclosures."
    ),
)

print("✅ Market Researcher Agent created")
print("✅ Quant Analyst Agent created")
print("✅ Manager Agent created")

✅ Market Researcher Agent created
✅ Quant Analyst Agent created
✅ Manager Agent created


## 4️⃣ Build Magentic Investment Research Workflow

### Configuration

| Method | Purpose |
|--------|---------|
| `.participants(...)` | Define available specialist agents |
| `.with_standard_manager(...)` | Configure orchestrator with AI Foundry |

In [4]:
print("\n🏦 Building Investment Research Workflow...")

# Build a sequential workflow: Manager → Market Researcher → Quant Analyst → Manager (synthesis)
# The manager orchestrates by first delegating to market researcher, then quant analyst
workflow = (
    WorkflowBuilder(start_executor=market_researcher, output_executors=[quant_analyst])
    .add_chain([market_researcher, quant_analyst])
    .build()
)

print("✅ Investment research workflow built successfully")


🏦 Building Investment Research Workflow...
✅ Investment research workflow built successfully


## 5️⃣ Define Investment Research Task

This quarterly research request requires:

1. **Market Research**: Current technology sector trends and market conditions
2. **Quantitative Analysis**: Calculate risk-adjusted returns and portfolio metrics
3. **Comparison**: Evaluate different investment approaches
4. **Synthesis**: Provide actionable recommendations with risk assessment

The orchestrator will automatically decompose this into subtasks and delegate to the appropriate specialists.

In [5]:
# Investment research request from a wealth advisor
research_request = (
    "I need a quarterly investment research brief for a client considering technology sector exposure. "
    "Please research the current state of the technology sector including recent performance of major indices "
    "(NASDAQ-100, S&P Technology Select Sector) and key market drivers. "
    "Then calculate the risk-adjusted return metrics for a hypothetical $100,000 investment allocated as: "
    "40% large-cap tech ETF (assume 12% annual return, 18% volatility), "
    "35% mid-cap growth fund (assume 15% annual return, 25% volatility), "
    "and 25% dividend tech stocks (assume 8% annual return, 12% volatility). "
    "Provide a summary table comparing expected returns, portfolio volatility, and Sharpe ratio "
    "(assuming 5% risk-free rate). Include a recommendation with risk considerations."
)

print(f"📋 Research Request: {research_request}")

📋 Research Request: I need a quarterly investment research brief for a client considering technology sector exposure. Please research the current state of the technology sector including recent performance of major indices (NASDAQ-100, S&P Technology Select Sector) and key market drivers. Then calculate the risk-adjusted return metrics for a hypothetical $100,000 investment allocated as: 40% large-cap tech ETF (assume 12% annual return, 18% volatility), 35% mid-cap growth fund (assume 15% annual return, 25% volatility), and 25% dividend tech stocks (assume 8% annual return, 12% volatility). Provide a summary table comparing expected returns, portfolio volatility, and Sharpe ratio (assuming 5% risk-free rate). Include a recommendation with risk considerations.


## 6️⃣ Execute Research Workflow with Streaming

The workflow execution:
1. Generates research plan (task decomposition)
2. Delegates to MarketResearcher for sector data
3. Delegates to QuantAnalyst for calculations
4. Monitors progress and adapts if needed
5. Synthesizes final research report
6. Completes when idle or max rounds reached

In [6]:
async def run_investment_research():
    """Run the investment research workflow with streaming output."""
    
    print("\n🚀 Starting investment research workflow...")
    print("=" * 60)
    
    # Run workflow and collect outputs
    outputs: list = []
    last_executor_id: str | None = None
    
    async for event in workflow.run(research_request, stream=True):
        # Capture streaming text from agents
        if event.type == "data" and isinstance(event.data, AgentResponseUpdate):
            eid = event.executor_id
            if eid != last_executor_id:
                if last_executor_id is not None:
                    print("\n")
                print(f"\n- {eid}:", end=" ", flush=True)
                last_executor_id = eid
            if event.data.text:
                print(event.data.text, end="", flush=True)
        
        # Capture final output
        elif event.type == "output":
            outputs.append(event.data)
    
    # Display final output
    print("\n\n" + "=" * 60)
    print("📊 INVESTMENT RESEARCH REPORT COMPLETE")
    print("=" * 60)
    
    if outputs:
        for output_data in outputs:
            if isinstance(output_data, list):
                for msg in output_data:
                    if hasattr(msg, 'text') and msg.text:
                        name = getattr(msg, 'author_name', None) or "assistant"
                        print(f"\n{'—' * 60}")
                        print(f"🤖 [{name.upper()}]")
                        print(f"{'—' * 60}")
                        print(msg.text)
            elif hasattr(output_data, 'text') and output_data.text:
                print(output_data.text)
            elif output_data:
                print(output_data)
    
    print("\n" + "=" * 60)
    print("⚠️ DISCLAIMER: This is for demonstration purposes only.")
    print("   Actual investment decisions require professional advice.")

## 7️⃣ Run the Investment Research Workflow

In [7]:
await run_investment_research()


🚀 Starting investment research workflow...


📊 INVESTMENT RESEARCH REPORT COMPLETE




Let
’s
 proceed
 with
 the
 **
calcul
ations
 for
 portfolio
 risk
-adjust
ed
 metrics
**
 and
 provide
 the
 full
 summary
 for
 the
 table
 and
 conclusion
.


---


###
 **
Portfolio
 Analysis
 &
 Risk
-
Adjusted
 Metrics
**


####
 
1
.
 **
Portfolio
 Expected
 Return
**
  

The
 portfolio
’s
 expected
 return
 is
 calculated
 as
 the
 weighted
 average
 of
 its
 components
:


\
[

E
(R
_p
)
 =
 w
_{\
text
{
ETF
}}
 \
cd
ot
 E
(R
_{\
text
{
ETF
}}
)
 +
 w
_{\
text
{
Growth
}}
 \
cd
ot
 E
(R
_{\
text
{
Growth
}}
)
 +
 w
_{\
text
{
Dividend
}}
 \
cd
ot
 E
(R
_{\
text
{
Dividend
}})

\
]


Where
:
  

-
 \(
 w
_{\
text
{
ETF
}}
 =
 
40
\
%
 \
),
 \(
 E
(R
_{\
text
{
ETF
}}
)
 =
 
12
\
%
 \
)
  

-
 \(
 w
_{\
text
{
Growth
}}
 =
 
35
\
%
 \
),
 \(
 E
(R
_{\
text
{
Growth
}}
)
 =
 
15
\
%
 \
)
  

-
 \(
 w
_{\
text
{
Dividend
}}
 =
 
25
\
%
 \
),
 \(
 E
(R
_{\
text
{
Dividend
}}
)
 =
 
8
\
%
 \
)




## 📝 Key Takeaways

### Magentic for Financial Research

| Benefit | Description |
|---------|-------------|
| **Intelligent Planning** | Orchestrator decomposes complex research requests |
| **Specialist Delegation** | Market researcher vs. quantitative analyst |
| **Adaptive Execution** | Adjusts plan based on findings |
| **Synthesized Reports** | Consolidated insights from multiple specialists |

### Industry Use Cases for Magentic Orchestration

| Use Case | Specialists Needed |
|----------|-------------------|
| Investment Research | Market Researcher + Quant Analyst |
| Due Diligence | Legal Analyst + Financial Analyst |
| Risk Assessment | Risk Analyst + Market Researcher |
| Client Reporting | Data Analyst + Compliance Writer |

### Magentic vs. Other Orchestration Patterns

| Pattern | When to Use | Key Feature |
|---------|-------------|-------------|
| **Concurrent** | Independent parallel tasks | Fan-out/fan-in |
| **Sequential** | Linear, dependent steps | Shared context |
| **Magentic** | Complex, adaptive decomposition | Intelligent orchestration |


### Production Considerations for FSI

| Consideration | Recommendation |
|---------------|----------------|
| **Compliance** | Log all orchestrator decisions for audit trail |
| **Cost Control** | Set appropriate max_round_count |
| **Data Currency** | Timestamp all market data references |
| **Disclaimers** | Include regulatory disclosures in final output |
| **Review** | Human review required before client delivery |